In [ ]:
import gower
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.dpi"] = 150
plt.style.use("ggplot")

mapping_dicts = {
    "Gender": {"0": "Male", "1": "Female", "0.0": "Male", "1.0": "Female"},
    "Job": {
        "1": "Unemployed",
        "2": "Employee/Worker",
        "3": "Manager/Executive",
        "4": "Entrepreneur/Freelancer",
        "5": "Retired",
    },
    "Area": {"1": "Nord", "2": "Centro", "3": "Sud/Isole"},
    "CitySize": {"1": "Small town", "2": "Medium-sized city", "3": "Large city"},
    "Investments": {
        "1": "No investments",
        "2": "Mostly lump sum",
        "3": "Capital accumulation",
    },
}

## Data Loading & Preprocessing


We extract the variables of interest, also dividing into:

- **Numerical**.
- **Categorical** (excluding the ID, which is not informative at this level).

We have to:

- **Encode categorical variables**, so that they can be digestible by clustering algorithms - they are all trasformed in boolean variables by one-hot encoding.
- **Normalize** in [0, 1] numerical variables.


In [ ]:
# Load data in a DataFrame
data = pd.read_excel("./Dataset1_BankClients.xlsx")

# Drop the column by its actual name (e.g., 'ID' or the actual name of the column)
data = data.drop(columns=["ID"])  # Replace 'ID' with the actual column name to drop

In [ ]:
# Specify categorical variables
categorical_columns = ["Gender", "Job", "Area", "CitySize", "Investments"]

# Apply one-hot encoding to categorical features
data_encoded = pd.get_dummies(data, columns=categorical_columns, dtype=int)

# Identify continuous numeric columns from the original data (exclude categorical columns).
continuous_columns = [
    col
    for col in data.select_dtypes(include=["number"]).columns
    if col not in categorical_columns
]

# Scale only continuous columns and keep one-hot columns unchanged.
scaler = StandardScaler()
data_scaled = data_encoded.copy()
data_scaled[continuous_columns] = scaler.fit_transform(data_scaled[continuous_columns])

data_scaled.head()

# KMedoids clustering

In the following code snippets:

- **We cluster our data into different k cluster**, k = 3, 4,... using **k-medoid**, as mentioned before.
- **We visualize clusters in a lower dimensional space for each value of k**(remember all the caveats of visualization in applying dimensionality reduction techniques).


In [ ]:
from sklearn_extra.cluster import KMedoids


# Function to perform k-medoids clustering and visualization
# K-medoids clustering with Gower distance
def perform_kmedoids_clustering(X, Y_3d, k, gower_distances):
    # Initialize and fit k-medoids
    kmedoids = KMedoids(
        n_clusters=k,
        metric="precomputed",
        random_state=42,
        init="k-medoids++",  # Changed from n_init
        max_iter=300,
    )

    labels = kmedoids.fit_predict(gower_distances)

    # 3D visualization
    # fig = plt.figure(figsize=(10, 8))
    # ax = fig.add_subplot(111, projection="3d")

    # colors = ["blue", "#4DBEEE", "#77AC30", "#EDB120", "#D95319", "#7E2F8E"]

    # for i in range(k):
    #     mask = labels == i
    #     ax.scatter(
    #         Y_3d[mask, 0],
    #         Y_3d[mask, 1],
    #         Y_3d[mask, 2],
    #         c=colors[i],
    #         edgecolor="k",
    #         alpha=0.8,
    #         label=f"Cluster {i + 1}",
    #     )

    # ax.set_title(f"3-D Embedding of {k} clusters")
    # ax.legend()
    # ax.view_init(elev=15, azim=-30)
    # plt.show()

    return labels

In [ ]:
# Clustering evaluation
def evaluate_clustering(X, cluster_results, gower_distances):
    k_values = list(cluster_results.keys())

    # Initialize metrics
    ch_scores = []
    db_scores = []
    sil_scores = []

    # Calculate metrics for each k
    for k in k_values:
        labels = cluster_results[k]
        ch_scores.append(calinski_harabasz_score(X, labels))
        db_scores.append(davies_bouldin_score(X, labels))
        sil_scores.append(
            silhouette_score(gower_distances, labels, metric="precomputed")
        )

    # Plot evaluation metrics
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

    # Calinski-Harabasz
    ax1.plot(k_values, ch_scores, "bo-")
    ax1.set_title("Calinski-Harabasz score")
    ax1.set_xlabel("Number of clusters")
    ax1.grid(True)

    # Davies-Bouldin
    ax2.plot(k_values, db_scores, "ro-")
    ax2.set_title("Davies-Bouldin score")
    ax2.set_xlabel("Number of clusters")
    ax2.grid(True)

    # Silhouette
    ax3.plot(k_values, sil_scores, "go-")
    ax3.set_title("Silhouette score")
    ax3.set_xlabel("Number of clusters")
    ax3.grid(True)

    plt.tight_layout()
    plt.show()

    # Print optimal values
    print(f"Optimal k according to Calinski-Harabasz: {k_values[np.argmax(ch_scores)]}")
    print(f"Optimal k according to Davies-Bouldin: {k_values[np.argmin(db_scores)]}")
    print(f"Optimal k according to Silhouette: {k_values[np.argmax(sil_scores)]}")

    return ch_scores, db_scores, sil_scores

In [ ]:
gower_distances = gower.gower_matrix(data_scaled)
tsne = TSNE(n_components=3, metric="precomputed", random_state=42, init="random")
Y_gower_3d = tsne.fit_transform(gower_distances)
cluster_results = {}
for k in range(3, 7):
    labels = perform_kmedoids_clustering(data_scaled, Y_gower_3d, k, gower_distances)
    cluster_results[k] = labels

In [ ]:
# Evaluate clustering results
ch_scores, db_scores, sil_scores = evaluate_clustering(
    data_scaled, cluster_results, gower_distances
)

# Bayesian GMM clustering


In [ ]:
import pandas as pd
from sklearn.mixture import BayesianGaussianMixture

# Set upper limit of candidate clusters
max_clusters = 15

# Train BGM with Dirichlet Process prior for auto-pruning
bgm = BayesianGaussianMixture(
    n_components=max_clusters,
    covariance_type="full",
    weight_concentration_prior_type="dirichlet_process",
    random_state=42,
    max_iter=1000,
    init_params="kmeans",
)

bgm.fit(data_scaled)

In [ ]:
# Raw cluster assignment from all components
raw_labels = bgm.predict(data_scaled)
raw_weights = pd.Series(bgm.weights_, name="weight")

# Lock final personas by removing low-weight components
weight_threshold = 0.05  # only keep components with > 5% weight
active_components = raw_weights[raw_weights > weight_threshold].index.tolist()

# Keep only samples assigned to active components
active_mask = pd.Series(raw_labels).isin(active_components)
data_persona = data_scaled.loc[active_mask].copy()
data_persona["persona_cluster"] = pd.Series(raw_labels)[active_mask].values

# Re-map cluster ids to consecutive labels (0..n-1) for readability
cluster_remap = {old: new for new, old in enumerate(sorted(active_components))}
data_persona["persona_cluster"] = data_persona["persona_cluster"].map(cluster_remap)

# Summary table of retained persona frameworks
persona_summary = (
    data_persona["persona_cluster"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("share")
    .to_frame()
)
persona_summary["count"] = (
    data_persona["persona_cluster"].value_counts().sort_index().values
)

print(f"Max components (K): {max_clusters}")
print(f"Retained persona frameworks: {len(active_components)}")
print(f"Active original component IDs: {sorted(active_components)}")

persona_summary

In [ ]:
data_persona

In [ ]:
# Save the clustered dataset into a new Excel file
data_persona.to_excel("result/bayesian-clustered-bankClients.xlsx", index=False)

# Qualitative Overlay: Rebuild Centroid Features into Business Tags

- Recover continuous centroid means in original units.
- Decode dominant one-hot categories for each retained cluster.
- Produce business-readable profiles to support persona naming.


In [ ]:
# Cluster means on scaled data
cluster_scaled_means = data_persona.groupby("persona_cluster").mean(numeric_only=True)

# Restore continuous means to original units
continuous_scaled = cluster_scaled_means[continuous_columns]
continuous_original = pd.DataFrame(
    scaler.inverse_transform(continuous_scaled),
    columns=continuous_columns,
    index=continuous_scaled.index,
)

# Decode dominant category from one-hot proportions
categorical_profile = pd.DataFrame(index=cluster_scaled_means.index)
for col in categorical_columns:
    onehot_cols = [c for c in cluster_scaled_means.columns if c.startswith(f"{col}_")]
    if not onehot_cols:
        continue

    proportions = cluster_scaled_means[onehot_cols]
    dominant_onehot = proportions.idxmax(axis=1)
    dominant_value_raw = dominant_onehot.str.replace(f"{col}_", "", regex=False)
    dominant_value_mapped = dominant_value_raw.map(
        lambda x: mapping_dicts.get(col, {}).get(str(x), x)
    )
    dominant_share = proportions.max(axis=1)

    categorical_profile[f"{col}_dominant"] = dominant_value_mapped
    categorical_profile[f"{col}_share"] = dominant_share

# Build base business profile and add cluster coverage metrics
business_profile = continuous_original.join(categorical_profile)
cluster_count = data_persona["persona_cluster"].value_counts().sort_index()
cluster_share = cluster_count / cluster_count.sum()
business_profile["persona_size"] = (
    cluster_count.reindex(business_profile.index).fillna(0).astype(int)
)
business_profile["persona_share"] = cluster_share.reindex(
    business_profile.index
).fillna(0.0)


# Categorical confidence helps flag whether dominant categories are clear or mixed.
def share_confidence_tag(v: float) -> str:
    if v >= 0.75:
        return "High confidence"
    if v >= 0.55:
        return "Medium confidence"
    return "Low confidence"


for col in categorical_columns:
    share_col = f"{col}_share"
    if share_col in business_profile.columns:
        business_profile[f"{col}_confidence"] = business_profile[share_col].apply(
            share_confidence_tag
        )

business_profile

## Final Clustering Result (5 Personas)

The final clustering solution identifies **5 customer personas** (total \(N = 4456\)).

### Persona 0: Mature Mainstream Working Families

- **Size:** 2288 clients (**51.35%**)
- **Profile:** Average age around 59, medium family size, medium income/wealth, moderate debt.
- **Behavior:** Balanced financial behavior, moderate digital usage, moderate financial education.
- **Dominant traits:** Female, Employee/Worker, Nord, Medium-sized city.
- **Investment style:** Capital accumulation.

### Persona 1: Urban Middle-Aged Female Accumulators

- **Size:** 704 clients (**15.80%**)
- **Profile:** Around 53 years old, smaller families, slightly higher income/wealth than Persona 0.
- **Behavior:** Higher digital adoption and more active lifestyle.
- **Dominant traits:** Female, mostly Employee/Worker, strongly Nord, Large city.
- **Investment style:** Capital accumulation.

### Persona 2: Elderly Male Retirees (Conservative)

- **Size:** 357 clients (**8.01%**)
- **Profile:** Oldest segment (about 81), small households, low income/wealth, very low debt.
- **Behavior:** Low digital engagement, low lifestyle/luxury orientation.
- **Dominant traits:** Male, Retired, Nord, Medium-sized city.
- **Investment style:** Mostly lump sum.

### Persona 3: Affluent Digital Professionals

- **Size:** 678 clients (**15.22%**)
- **Profile:** Around 52 years old, highest income/wealth among all clusters.
- **Behavior:** Highest financial education, strongest digital adoption, high ESG and lifestyle/luxury orientation.
- **Dominant traits:** Mostly Male, Employee/Worker, strongly Nord, Large city.
- **Investment style:** Capital accumulation.

### Persona 4: Elderly Female Retirees (Low-Intensity Banking)

- **Size:** 429 clients (**9.63%**)
- **Profile:** About 82 years old, low income/wealth, low debt.
- **Behavior:** Low digital activity and low lifestyle/luxury propensity.
- **Dominant traits:** Female, Retired, Nord, Medium-sized city.
- **Investment style:** Mostly lump sum.

### Overall Interpretation

The clustering is mainly driven by:

1. **Life stage** (working-age vs retired)
2. **Economic capacity** (mass market vs affluent)
3. **Behavioral engagement** (digital/active vs low-intensity)
4. **Retired segment split by gender** (Persona 2 vs Persona 4)

This segmentation provides a strong basis for targeted strategies in advisory, digital engagement, and product personalization.
